# Two-Stage Photogrammetry Pipeline: Coarse-to-Fine

## Concept Overview
This pipeline is designed for high-quality 3D reconstruction of objects (like a house) using a **Coarse-to-Fine** approach.

1.  **Stage 1 (Coarse/Scout):** We use standard, fast algorithms (SIFT + COLMAP) on downsampled images to quickly estimate camera poses and the geometry of the scene. The goal here is **not** pretty details, but to understand *where* the object is in 3D space.
2.  **Geofencing:** Using the coarse model, we calculate a 3D Bounding Box (ROI). We generate 2D masks for the images so subsequent steps ignore the background (sky, distant trees).
3.  **Stage 2 (Fine/Refined):** We use State-of-the-Art Deep Learning extractors (**SuperPoint**) and matchers (**LightGlue**). We assume the poses from Stage 1 are roughly correct to guide the matching, but we refine them. This creates a highly accurate sparse point cloud.
4.  **Dense Reconstruction (OpenMVS):** We convert the data to OpenMVS format to generate a dense point cloud, a mesh, and finally a texture.
5.  **Gaussian Splatting Prep:** The output of Stage 2 (Undistorted Sparse Model) is perfectly formatted to train 3D Gaussian Splats.

---

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import shutil
import subprocess
import logging
from pathlib import Path
import numpy as np
import cv2
from PIL import Image
import pycolmap

# HLOC Imports
from hloc import (
    extract_features,
    match_features,
    reconstruction,
    triangulation,
    pairs_from_poses,
    # The following alternatives are available, try them if initial Pose Estimation (Stage 1) or GPS Priors is poor
        # pairs_from_exhaustive,
        # pairs_from_covisibility,
    visualization,
    extract_gps
)
from hloc.utils.io import read_image, run_command

# Setup Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("Pipeline")

## 1. Configuration & Paths
Define all your parameters here. This makes the notebook easy to rerun for different experiments.

In [ ]:
from typing import Literal

Extractor = Literal["superpoint_max", "superpoint_aachen", "disk"]
Matcher = Literal["lightglue", "superglue"]
PairMode = Literal["pose", "exhaustive", "covisibility"]

class Config:
    # --- Dataset ---
    experiment = "FriedrichsHouse"
    dataset_root = Path("/data/")
    raw_images = dataset_root / experiment / "full"
    
    # --- Output ---
    output_root = Path(f"/data/output/hloc/{experiment}")
    
    # --- Stage 1 (Coarse) Settings ---
    # Uses SIFT (via pycolmap) for speed and robustness
    coarse_max_img_size = 1600
    coarse_num_features = 2048
    prior_std_xy = 2.0  # GPS accuracy priors
    prior_std_z = 3.0

    # --- Stage 2 (Fine) Settings ---
    # Uses SuperPoint+LightGlue for precision
    # fine_max_img_size = 1600 # Downscale for SfM math (faster)
    fine_extractor: Extractor = "superpoint_max" 
    fine_matcher: Matcher = "lightglue"
    _fine_extractor_conf = extract_features.confs[fine_extractor]
    fine_max_img_size: int = _fine_extractor_conf["preprocessing"]["resize_max"] # To be set based on extractor config
    # "pose" pairs means we only match images that look at the same thing based on Stage 1
    pair_mode: PairMode = "pose" 
    neighbors_to_match = 15 # How many neighbors from Stage 1 to match in Stage 2 (top k-views)

    # --- Geofencing ---
    # Defines the "Cylinder" around the object to keep
    silo_radius_ratio = 0.4
    
    # --- MVS / Meshing ---
    openmvs_bin = "/usr/local/bin/OpenMVS/" # Path to OpenMVS binaries
    mvs_scale = 1.0 # 1.0 = Use full resolution images for texturing, 0.5 = Half res

# Ensure output directories exist
Config.output_root.mkdir(parents=True, exist_ok=True)

# Derived Paths
paths = {
    "coarse_sfm": Config.output_root / "sfm_coarse",
    "coarse_db": Config.output_root / "sfm_coarse" / "database.db",
    "fine_sfm": Config.output_root / f"sfm_{Config.fine_extractor}+{Config.fine_matcher}",
    "fine_db": Config.output_root / f"sfm_{Config.fine_extractor}+{Config.fine_matcher}" / "database.db",
    "resized_images": Config.raw_images / "resized",
    "raw_masks": Config.raw_images / "masks_geo",
    "masks": Config.raw_images / "resized" / "masks_geo",
    "features": Config.output_root / "features.h5",
    "matches": Config.output_root / "matches.h5",
    "pairs": Config.output_root / "pairs.txt",
    "mvs_root": Config.output_root / "mvs_workspace"
}

for p in paths.values():
    if p.suffix == "": # If it's a directory
        p.mkdir(parents=True, exist_ok=True)

print(f"Experiment initialized: {Config.experiment}")

## 2. Utilities
Wrapper functions to handle subprocesses cleanly and manage the environment.

In [ ]:
def run_cmd(command, cwd=None, env=None):
    """Run a subprocess command with live output streaming."""
    if cwd:
        cwd = str(cwd)
    
    cmd_str = " ".join(command)
    logger.info(f"Running: {cmd_str}")

    process = subprocess.Popen(
        command, 
        stdout=subprocess.PIPE, 
        stderr=subprocess.STDOUT, 
        text=True, 
        bufsize=1, 
        cwd=cwd,
        env=env
    )

    for line in process.stdout:
        print(line, end='')

    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f"Command failed with code {process.returncode}: {cmd_str}")

class OpenMVSEnv:
    """Context manager to temporarily add OpenMVS to PATH."""
    def __enter__(self):
        self.old_env = os.environ.copy()
        os.environ["PATH"] = f"{Config.openmvs_bin}:{os.environ['PATH']}"
        return os.environ

    def __exit__(self, exc_type, exc_value, traceback):
        os.environ.clear()
        os.environ.update(self.old_env)

def scale_intrinsics(camera, scale):
    """Scales COLMAP camera intrinsics."""
    camera.width = int(camera.width * scale)
    camera.height = int(camera.height * scale)
    
    # Focal length indices vary by model, but usually 0 or 0 and 1
    if camera.model in ["SIMPLE_PINHOLE", "SIMPLE_RADIAL", "RADIAL"]:
        camera.params[0] *= scale # f
        camera.params[1] *= scale # cx
        camera.params[2] *= scale # cy
    elif camera.model in ["PINHOLE", "OPENCV", "FULL_OPENCV"] or camera.focal_length_idxs == 2:
        camera.params[0] *= scale # fx
        camera.params[1] *= scale # fy
        camera.params[2] *= scale # cx
        camera.params[3] *= scale # cy
    return camera

## 3. Stage 1: Coarse Reconstruction (The Scout)

**Goal:** Fast, rough estimate of camera poses.
**Theory:** We use SIFT (Scale Invariant Feature Transform). It's older but extremely mathematically robust and fast to extract via GPU. We use standard COLMAP sequential matching here to build a "skeleton" of the scene.

In [ ]:
# -----------------------------------------------------------------------------
# [OPTIONAL] RESET STAGE 1
# Set this to True if you want to completely erase the Coarse Reconstruction
# and start the Scout step from scratch.
# -----------------------------------------------------------------------------
RESET_STAGE_1 = True 

if RESET_STAGE_1:
    target_dir = paths["coarse_sfm"]
    
    # Safety Check: Ensure the target directory is actually inside our defined output folder
    # This prevents accidental deletion of raw images or root directories.
    if Config.output_root in target_dir.resolve().parents:
        if target_dir.exists():
            logger.warning(f"🧹 RESETTING STAGE 1: Deleting directory {target_dir}...")
            shutil.rmtree(target_dir)
            
            # Re-create the empty directory so the next step doesn't fail
            target_dir.mkdir(parents=True, exist_ok=True)
            logger.info("Stage 1 artifacts cleared.")
        else:
            logger.info(f"Stage 1 directory ({target_dir}) does not exist. Nothing to clean.")
    else:
        logger.error(f"🛑 SAFETY STOP: Attempted to delete {target_dir}, but it is not inside the Experiment Output Root. Deletion cancelled.")

In [ ]:
if not (paths["coarse_sfm"] / "0" / "cameras.bin").exists():
    logger.info("Starting Coarse Feature Extraction...")

    # Crates a list of image names in images (dir) without subdirs
    images_list = [f.name for f in Config.raw_images.iterdir() if f.is_file()]
    
    # 1. Extract Features (SIFT via COLMAP)
    pycolmap.extract_features(
        paths["coarse_db"], 
        image_path = Config.raw_images, 
        image_names= images_list,
        camera_model="SIMPLE_RADIAL", 
        # Create one camera model per folder - incase of one drone with one camera this assumes all images where taken with same camera with same intrinstics (focal length, etc)
        # Use CameraMode.AUTO if multiple cameras are used
        camera_mode=pycolmap.CameraMode.PER_FOLDER, 
        extraction_options={
            "max_image_size": Config.coarse_max_img_size,
            "sift": {"max_num_features": Config.coarse_num_features}
        },
        device="cuda"
    )

    # 2. Match (Sequential/Spatial is usually enough for a sequence)
    logger.info("Starting Coarse Matching...")
    num_images: int = 0
    with pycolmap.Database.open(paths["coarse_db"]) as db:
        num_images = db.num_images()


    # Asign matching options
    # For using pycolamp matching functions
    matching_options = {    "use_gpu": True,
                            "max_num_matches": 32768,
                            "sift": {
                                "max_ratio": 0.8, 
                                "max_distance": 0.7, 
                                "cross_check": True, 
                                "cpu_brute_force_matcher": False        
                            },
                        }
    # Equivalent COLMAP CLI command
    cmd = [
            "colmap", "spatial_matcher",
            "--database_path", str(paths["coarse_db"]),
            "--FeatureMatching.use_gpu", "1",
            "--FeatureMatching.gpu_index", "-1", # Or assign specific GPU
            "--FeatureMatching.max_num_matches", "32768", # Increase max matches for better coverage or lower for speed
            "--SiftMatching.cross_check", "1",
            "--SiftMatching.cpu_brute_force_matcher", "0",
        ]

    # Using exhaustive if dataset < 60 images, otherwise spatial
    if num_images < 60:
        pycolmap.match_exhaustive(paths["coarse_db"], matching_options=matching_options, device="cuda")
    else:
        cmd += [
            "--SpatialMatching.ignore_z", "1", # Ignore altitude for matching - all images taken from drone at similar altitude
        ]
        # pycolmap.match_spatial(paths["coarse_db"], matching_options=matching_options, device="cuda")
        run_cmd(cmd)

    # 3. Reconstruct (Mapper)
    logger.info("Starting Coarse Mapping...")
    # Note: We incorporate GPS priors here to align the world correctly immediately
    # This saves us from having to manually align the model later.
    # requires COLMAP CLI because pycolmap mapper bindings are basic
    cmd = [
        "colmap", "pose_prior_mapper",
        "--database_path", str(paths["coarse_db"]),
        "--image_path", str(Config.raw_images),
        "--output_path", str(paths["coarse_sfm"]),
        "--prior_position_std_x", str(Config.prior_std_xy),
        "--prior_position_std_y", str(Config.prior_std_xy),
        "--prior_position_std_z", str(Config.prior_std_z),
        "--overwrite_priors_covariance", "1",
    ]
    run_cmd(cmd)
else:
    logger.info("Coarse model found. Skipping Stage 1.")

## 4. Geofencing & Mask Generation

**Goal:** Create binary masks to block out the sky and neighbors.
**Theory:** MVS algorithms try to match everything. Sky and distant moving trees cause noise (artifacts) in the mesh. By using the coarse model, we know where the cameras are and where the dense point cloud roughly is. We calculate a 3D bounding box around the "Silo" (the house) and project this box into every camera view to create a mask.

In [ ]:
# -----------------------------------------------------------------------------
# [OPTIONAL] RESET SECTION 4 (MASKS)
# Set this to True to delete the masks generated from the Coarse Model.
# -----------------------------------------------------------------------------
RESET_MASKS = True

# We define the path explicitly here to ensure we are targeting the 
# RAW resolution masks, not the resized ones.

if RESET_MASKS:
    # Safety Check:
    # 1. Ensure the folder name is exactly "masks_geo"
    # 2. Ensure it is a subdirectory of the dataset root
    is_safe_name = paths["raw_masks"].name == "masks_geo"
    is_inside_dataset = Config.dataset_root in paths["raw_masks"].resolve().parents
    
    if is_safe_name and is_inside_dataset:
        if paths["raw_masks"].exists():
            logger.warning(f"🧹 RESETTING MASKS: Deleting directory {paths['raw_masks']}...")
            shutil.rmtree(paths["raw_masks"])
            logger.info("Raw geofencing masks cleared.")
        else:
            logger.info(f"Mask directory ({paths['raw_masks']}) does not exist. Nothing to clean.")
    else:
        logger.error(f"🛑 SAFETY STOP: Attempted to delete {paths['raw_masks']}. Safety check failed.")

In [ ]:
from hloc import geofencing, masking_geo

# 1. Load Coarse Model
# This model was built using the Raw Images, so its cameras have full resolution intrinsics.
coarse_model = pycolmap.Reconstruction(paths["coarse_sfm"] / "0")

# 2. Calculate Adaptive Bounding Box
bbox_min, bbox_max = geofencing.compute_adaptive_geofence(
    coarse_model, 
    silo_ratio=Config.silo_radius_ratio, 
    height_center_bias=0.8, 
    safety_margin=0.1
)

# 3. Create Masks at RAW Resolution
# We store these next to the raw images because they match the raw image dimensions.


if not paths["raw_masks"].exists():
    logger.info(f"Generating masks into {paths['raw_masks']}...")
    
    # We pass the target folder directly. 
    # HLOC will write {image_name}.png inside this folder.
    ret = masking_geo.create_mvs_masks(
        model=coarse_model,
        output_mask_folder=paths["raw_masks"], 
        bbox_min=bbox_min,
        bbox_max=bbox_max,
    )
    
    # --- HLOC Path Handling Fix ---
    # Some versions of HLOC's create_mvs_masks might create a subfolder 
    # named 'masks_geo' *inside* the folder you provided. 
    # We check for this nesting and fix it if it happens.
    nested_folder = paths["raw_masks"] / "masks_geo"
    if nested_folder.exists() and nested_folder.is_dir():
        logger.info("Detected nested 'masks_geo' folder created by HLOC. Fixing structure...")
        for file in nested_folder.iterdir():
            shutil.move(str(file), str(paths["raw_masks"]))
        nested_folder.rmdir()
        
    logger.info("Mask generation complete.")
else:
    logger.info(f"Masks already exist at {paths['raw_masks']}. Skipping generation.")



## 5. Stage 2: Fine Reconstruction (The Artist)

**Goal:** High-precision sparse point cloud.
**Theory:**
*   **Resizing:** Deep learning extractors work on fixed grids. 1600px is a sweet spot for SuperPoint.
*   **SuperPoint:** A neural network trained to find corners and blobs that are trackable across viewpoint changes.
*   **LightGlue:** A "graph neural network" matcher. It looks at the whole constellation of points in Image A and Image B and filters out outliers based on geometric consistency (it understands context).
*   **Pairs from Poses:** Instead of matching Image 1 to Image 500 (which is impossible), we ask the Coarse model: "Which images look at the same part of the house?" and only match those.

In [ ]:
# -----------------------------------------------------------------------------
# [OPTIONAL] RESET RESIZED IMAGES & MASKS
# Set this to True to delete all downscaled images and masks.
# Useful if you changed the target resolution (fine_max_img_size).
# -----------------------------------------------------------------------------
RESET_RESIZED = True

if RESET_RESIZED:
    folders_to_clean = [paths["resized_images"], paths["masks"]]
    
    for folder in folders_to_clean:
        # Safety Check: Ensure we are deleting a 'resized' folder, not the raw data
        if folder.exists() and "resized" in str(folder):
            logger.warning(f"🧹 RESETTING RESIZED DATA: Deleting {folder}...")
            shutil.rmtree(folder)
            folder.mkdir(parents=True, exist_ok=True) # Recreate empty dir
        elif not folder.exists():
            logger.info(f"Directory {folder} does not exist. Skipping.")
        else:
            logger.error(f"🛑 SAFETY STOP: Skipped deletion of {folder} because it does not look like a 'resized' directory.")
    
    logger.info("Resized images and masks cleared.")

In [ ]:
# 1. Resize Images and Masks for Processing
from hloc.extract_features import resize_image

def process_resize(src_dir, dst_dir, max_size, interpolation="cv2_area", ext_filter=[".jpg", ".png"]):
    dst_dir.mkdir(parents=True, exist_ok=True)
    files = [f for f in src_dir.iterdir() if f.suffix.lower() in ext_filter]

    if not files:
        logger.warning(f"No files found in {src_dir} with extensions {ext_filter}. Skipping resize.")
        return 1.0
    
    # NOTE: Current implementation assumes all raw images have the SAME dimensions.
    # The scale is calculated once based on the first image found.
    # Future improvement: Store scale factors per image (e.g., dict[filename, scale]) 
    # to support datasets with varying image sizes and allow precise per-image upscaling later.
    img = read_image(files[0])
    size = img.shape[:2][::-1]
    scale = max_size / max(size) if max(size) > max_size else 1.0
    logger.info(f"Resizing images from {src_dir} to max size {max_size}px with scale factor {scale:.4f}...")

    for f in files:
        if (dst_dir / f.name).exists(): continue
        
        if interpolation == "cv2_nearest":
            # Special handling for masks (must remain binary 0 or 255)
            img = read_image(f, grayscale=True)
            interp = "cv2_nearest"
        else:
            img = read_image(f)
            interp = "cv2_area"
            
        size = img.shape[:2][::-1]
        if max(size) > max_size:
            _scale = max_size / max(size)
            new_size = tuple(int(round(x * _scale)) for x in size)
            img = resize_image(img, new_size, interp=interp)
        
        cv2.imwrite(str(dst_dir / f.name), img if len(img.shape)==2 else img[:,:,::-1])

    return scale


logger.info("Resizing Images...")
Config.global_scale = process_resize(Config.raw_images, paths["resized_images"], Config.fine_max_img_size)
logger.info(f"Global Scale Factor applied to images: {Config.global_scale:.4f}")

logger.info("Resizing Masks...")
# Masks usually need nearest neighbor to avoid gray edges
process_resize(paths["raw_masks"], paths["masks"], Config.fine_max_img_size, interpolation="cv2_nearest", ext_filter=[".png"])

# Rename masks for OpenMVS (needs .mask.png extension)
for m in paths["masks"].glob("*.png"):
    if not m.name.endswith(".mask.png"):
        m.rename(m.with_suffix(".mask.png"))

### Extract & Match Features 

In [ ]:
# -----------------------------------------------------------------------------
# [OPTIONAL] RESET FEATURES & MATCHES
# Set these to True to force re-extraction and re-matching of features.
# -----------------------------------------------------------------------------
RESET_FEATS = True
RESET_MATCHES = True

In [ ]:
# 2. Extract SuperPoint Features
feature_conf = extract_features.confs[Config.fine_extractor]
feature_path = extract_features.main(
    feature_conf, 
    paths["resized_images"], 
    feature_path=paths["features"],
    mask_dir=paths["masks"], # Apply masks during extraction to ignore sky keypoints
    overwrite=RESET_FEATS
)

# 3. Generate Pairs from Coarse Poses
pairs_from_poses.main(
    paths["coarse_sfm"] / "0", 
    paths["pairs"], 
    num_matched=Config.neighbors_to_match
)

# 4. Match Features (LightGlue)
matcher_conf = match_features.confs["superpoint+lightglue"]
match_path = match_features.main(
    matcher_conf, 
    paths["pairs"], 
    features=paths["features"], 
    matches=paths["matches"],
    overwrite=RESET_MATCHES
)

In [ ]:
# -----------------------------------------------------------------------------
# [OPTIONAL] RESET STAGE 2
# Set this to True if you want to completely erase the Fine Reconstruction
# and start the Scout step from scratch.
# -----------------------------------------------------------------------------

RESET_STAGE_2 = True

if RESET_STAGE_2:
    target_dir = paths["fine_sfm"]
    
    # Safety Check: Ensure the target directory is actually inside our defined output folder
    # This prevents accidental deletion of raw images or root directories.
    if Config.output_root in target_dir.resolve().parents:
        if target_dir.exists():
            logger.warning(f"🧹 RESETTING STAGE 2: Deleting directory {target_dir}...")
            shutil.rmtree(target_dir)
            
            # Re-create the empty directory so the next step doesn't fail
            target_dir.mkdir(parents=True, exist_ok=True)
            logger.info("Stage 2 artifacts cleared.")
        else:
            logger.info(f"Stage 2 directory ({target_dir}) does not exist. Nothing to clean.")
    else:
        logger.error(f"🛑 SAFETY STOP: Attempted to delete {target_dir}, but it is not inside the Experiment Output Root. Deletion cancelled.")

In [ ]:
# 5. Sparse Reconstruction

if not (paths["fine_sfm"] / "0" / "cameras.bin").exists():
    # Initialize new DB
    if paths["fine_db"].exists(): paths["fine_db"].unlink()
    reconstruction.create_empty_db(paths["fine_db"])
    
    # Import Images
    # We point to the resized images + the masks
    reconstruction.import_images(
        paths["resized_images"], 
        paths["fine_db"], 
        pycolmap.CameraMode.PER_FOLDER, 
        image_list= images_list,
        options=pycolmap.ImageReaderOptions({"mask_path": paths["masks"]})
    )
    
    # Import Features/Matches
    image_ids = reconstruction.get_image_ids(paths["fine_db"])
    with pycolmap.Database.open(paths["fine_db"]) as db:
        logger.info(f"Number of images in Fine DB: {db.num_images()}")
        triangulation.import_features(image_ids, db, paths["features"])
        triangulation.import_matches(image_ids, db, paths["pairs"], paths["matches"], skip_geometric_verification=False)
    
        # Triangulate / Verify
        triangulation.estimation_and_geometric_verification(paths["fine_db"], paths["pairs"])
    
    # Re-inject GPS Priors (using original image metadata)
    extract_gps.populate_priors(paths["fine_db"], Config.raw_images)
    
    # Run Mapper with Priors
    logger.info("Running Fine Mapper...")
    cmd = [
        "colmap", "pose_prior_mapper",
        "--database_path", str(paths["fine_db"]),
        "--image_path", str(paths["resized_images"]),
        "--output_path", str(paths["fine_sfm"]),
        "--prior_position_std_x", str(Config.prior_std_xy),
        "--prior_position_std_y", str(Config.prior_std_xy),
        "--prior_position_std_z", str(Config.prior_std_z),
        "--overwrite_priors_covariance", "1"
    ]
    run_cmd(cmd)

In [ ]:
# 6. Final Clean: Geometric Crop of Sparse Model
# This ensures the sparse model used for Gaussian Splatting doesn't have floaters
final_model = pycolmap.Reconstruction(paths["fine_sfm"] / "0")
geofenced_path = paths["fine_sfm"] / "geofenced"
geofenced_path.mkdir(exist_ok=True)

cropped_model_geofence_specs = geofencing.pca_cylinder_geofence(
    final_model,
    output_model_path=geofenced_path,
    buffer_dist=0.0
)
print(f"Final Sparse Model ready at: {geofenced_path}")

## 6. Dense Reconstruction (OpenMVS)

**Goal:** A watertight mesh with high-resolution texture.
**Theory:**
*   **InterfaceCOLMAP:** Converts the sparse point cloud and camera poses into OpenMVS format (`.mvs`).
*   **Densify:** Computes depth maps for every image and fuses them. This creates millions of points.
*   **Mesh:** Creates a surface from the points. We use **Poisson** from **PyMeshlab** initially, then **Refine** it using **OpenMVS** *RefineMesh*.
*   **Texture:** Projects the original images onto the mesh to create a UV map. We use [MVS-texturing tool](https://github.com/nmoehrle/mvs-texturing.git) to create textures from the highres-images

In [ ]:
# -----------------------------------------------------------------------------
# RESET MVS WORKSPACE
# Set this to True to delete all MVS workspace data.
# -----------------------------------------------------------------------------
RESET_MVS = True

if RESET_MVS:
    target_dir = paths["mvs_root"]
    has_mvs_name = "mvs" in target_dir.name.lower()
    
    # Safety Check: Ensure the target directory is actually inside our defined output folder
    # This prevents accidental deletion of raw images or root directories.
    if Config.output_root in target_dir.resolve().parents and has_mvs_name:
        if target_dir.exists():
            logger.warning(f"🧹 RESETTING MVS WORKSPACE: Deleting directory {target_dir}...")
            shutil.rmtree(target_dir)
            
            # Re-create the empty directory so the next step doesn't fail
            target_dir.mkdir(parents=True, exist_ok=True)
            logger.info("MVS workspace cleared.")
        else:
            logger.info(f"MVS workspace directory ({target_dir}) does not exist. Nothing to clean.")
    else:
        logger.error(f"🛑 SAFETY STOP: Attempted to delete {target_dir}, but it is not inside the Experiment Output Root. Deletion cancelled.")

In [ ]:
# 1. Undistort Images (Prepare for MVS)
# OpenMVS expects undistorted pinhole images.
undistorted_path = paths["mvs_root"] / "images"
pycolmap.undistort_images(
    paths["mvs_root"], 
    geofenced_path, 
    paths["resized_images"], 
    output_type="COLMAP"
)

In [ ]:
# 2. Prepare High-Res Texture Project (Optional but recommended)
# We create a second .mvs file that uses the original high-res images, 
# but shares the geometry from the low-res processing.

logger.info("Creating Scaled Reconstruction for Texturing...")
scaled_model_path = paths["mvs_root"] / "sparse_scaled"
scaled_model_path.mkdir(exist_ok=True)

model = pycolmap.Reconstruction(paths["mvs_root"] / "sparse")
# Calculate scale factor relative to the images used in Stage 2
# If we used 1600px for SfM, and originals are 4000px, scale is ~2.5
# Here we want to go back to ORIGINALS.

img_name = list(model.images.values())[0].name
orig_w, orig_h = Image.open(Config.raw_images / img_name).size
sfm_w = list(model.cameras.values())[0].width

scale_factor = orig_w / sfm_w
logger.info(f"Scaling model by factor: {scale_factor}")

for cam in model.cameras.values():
    scale_intrinsics(cam, scale_factor)

model.write(scaled_model_path)

# Undistort Original High-Res Images
pycolmap.undistort_images(
    scaled_model_path, 
    scaled_model_path, 
    Config.raw_images, 
    output_type="COLMAP",
    # output_image_dir=paths["mvs_root"] / "images_highres"
)

### Dense Model

In [ ]:
# 3. OpenMVS Pipeline
with OpenMVSEnv() as env:
    # A. Convert COLMAP -> MVS
    # We use the scaled model (high res intrinsics) but point to the resized images folder temporarily
    # We will swap the image folder later for texturing, or OpenMVS handles finding the originals if names match.
    run_cmd([
        "InterfaceCOLMAP",
        "-i", ".",
        "-o", "scene.mvs",
        "--image-folder", str(paths["mvs_root"] / "images"), # Use undistorted ones first
        "--archive-type", "1",
        "--common-intrinsics", "1",
    ], cwd=paths["mvs_root"], env=env)

    # B. Densify
    # Uses the masks we generated earlier to avoid densifying sky
    run_cmd([
        "DensifyPointCloud",
        "-i", "scene.mvs",
        "-o", "scene_dense.mvs",
        "--max-resolution", str(Config.fine_max_img_size),
        "--min-resolution", "640",
        "--sub-resolution-levels", "2",
        "--postprocess-dmaps", "1",
        "--fusion-mode", "0", # 0=Global (better for small objects), -1=DepthMap (better for large scenes)
        "--mask-path", str(paths["masks"])
    ], cwd=paths["mvs_root"], env=env)

    # C. Mesh
    """ run_cmd([
        "ReconstructMesh",
        "scene_dense.mvs",
        "-p", "scene_dense.ply",
        "--min-point-distance", "2.5", # 2. MERGE CLOSE POINTS (Prevents math errors/segfaults)
        "--remove-spurious", "40",   # 3. REMOVE NOISE (Cleans outlier points)
        "--free-space-support", "0", # 4. SIMPLIFY CALCULATION (Disable complex free-space logic)
        "-o", "scene_mesh.mvs"
    ], cwd=paths["mvs_root"], env=env) """
    

### Meshing

In [ ]:
import pymeshlab
ms = pymeshlab.MeshSet()
ms.load_new_mesh(str((paths["mvs_root"]/"scene_dense.ply").resolve()))

ms.generate_surface_reconstruction_screened_poisson(depth=11, preclean=True) # 12 for more details, requires more memory


In [ ]:
# ---------------------------------------------------------
# FIX: CROP TO BOUNDING BOX
# ---------------------------------------------------------
print("2. Cropping Mesh to Geofence...")

# Select vertices OUTSIDE the box
# --- INPUTS FROM YOUR PREVIOUS STEP ---

c_center = cropped_model_geofence_specs["center"]
c_axis  = cropped_model_geofence_specs["normal"]
c_radius = cropped_model_geofence_specs["radius"]


# Normalize axis to be safe (crucial for the math to work)
c_axis = c_axis / np.linalg.norm(c_axis)

# --- PATHS ---
input_ply = str((paths["mvs_root"] / "scene_dense.ply").resolve())
output_mesh = str((paths["mvs_root"] / "mesh_poisson.ply").resolve())

# --- GENERATE MESHLAB CONDITION STRING ---
# We need to select vertices where the distance to the axis line is > radius.
# Math: Dist_sq = ||(P - Center) - ((P - Center) . Axis) * Axis||^2

# 1. Define vectors relative to Center (P - C)
vx = f"(x - {c_center[0]})"
vy = f"(y - {c_center[1]})"
vz = f"(z - {c_center[2]})"

# 2. Dot Product: (P - C) . Axis
# Note: We simply multiply the components. 
dot = f"({vx} * {c_axis[0]} + {vy} * {c_axis[1]} + {vz} * {c_axis[2]})"

# 3. Perpendicular Vector components: V_perp = V - (Dot * Axis)
perp_x = f"({vx} - {dot} * {c_axis[0]})"
perp_y = f"({vy} - {dot} * {c_axis[1]})"
perp_z = f"({vz} - {dot} * {c_axis[2]})"

# 4. Final Condition: Distance Squared > Radius Squared
# We use perp_x * perp_x instead of ^2 to ensure compatibility across versions
cyl_condition = f"({perp_x}*{perp_x} + {perp_y}*{perp_y} + {perp_z}*{perp_z}) > {c_radius**2}"

print("2. cutting Mesh with Cylinder...")
# Select vertices outside the cylinder
ms.compute_selection_by_condition_per_vertex(condselect=cyl_condition)
# Remove the selected vertices (and the faces attached to them)
ms.meshing_remove_selected_vertices()

print("3. Removing the 'Skirt' (Stretched Edges)...")
# This is the key fix. Stretched texture = Long Triangles.
# We select faces with edges longer than a threshold.
# Note: 'threshold' is in your world units (meters). 
# If your house is ~10m wide, a triangle edge of 2.0m is likely garbage/sky.
ms.compute_selection_by_edge_length(threshold=1.5) 
ms.meshing_remove_selected_faces()

print("4. Cleaning up floating bits...")
# Remove small disconnected pieces that are floating in the air
ms.meshing_remove_connected_component_by_diameter(mincomponentdiag=pymeshlab.PureValue(2.0))
ms.meshing_remove_unreferenced_vertices()

print("5. Closing Holes...")
# We use maxholesize to close windows/roof gaps, but NOT the huge bottom opening.
# 1000 is an arbitrary number of faces. Increase if windows are still open.
ms.meshing_close_holes(maxholesize=1000) 

print(f"6. Saving to {((paths['mvs_root'] / 'mesh_poisson.ply').resolve())}...")
ms.save_current_mesh(str((paths["mvs_root"]/'mesh_poisson.ply').resolve()))

In [ ]:
with OpenMVSEnv() as env:
    # D. Refine Mesh
    # This improves detail by using image consistency
    run_cmd([
        "RefineMesh",
        "scene_dense.mvs",
        "-m" , "mesh_poisson.ply",
        "-o", "scene_mesh_refined.mvs",
        "--scales", "2",
        "--max-face-area", "32",
        "--min-resolution", str(Config.fine_max_img_size // 4),
        "--reduce-memory", "0",
        "--planar-vertex-ratio", "0.5",
        "--regularity-weight", "0.5",
        "--decimate", "0.5", # Reduce to x% of original number of vertices before refining for speed
        "--cuda-device", "-1" # Use best GPU available BUT Often more stable on CPU for complex meshes
    ], cwd=paths["mvs_root"], env=env)

    # E. Texture
    # Here we should point to original images if possible for highest res
    # For simplicity, we use the ones we have, but ensure Patch Packing is optimal
    """ run_cmd([
        "TextureMesh",
        "scene_mesh_refined.mvs",
        "-o", "textured_model.glb", # Export directly to GLB for web viewing
        "--patch-packing-heuristic", "0"
    ], cwd=paths["mvs_root"], env=env) """

### Texturing 

In [ ]:
# 4. Export to VisualSFM NVM Format for Texturing with MVS texturing tool
with OpenMVSEnv() as my_env:
    run_cmd([
            "colmap", "model_converter",
            "--input_path", str(scaled_model_path),
            "--output_path", str(scaled_model_path / "sfm_model.nvm"),
            "--output_type", "NVM"
        ], cwd=str(scaled_model_path), env=my_env),

In [ ]:
# 5. FIX NVM PATHS
logger.info("Fixing NVM file paths for MVS Texturing...")

from pathlib import Path

# Configuration
nvm_path = scaled_model_path / "sfm_model.nvm"
image_subdir = "images"  # The folder name where your images are stored

print(f"Reading NVM: {nvm_path}")

with open(nvm_path, 'r') as f:
    lines = f.readlines()

# --- NVM Format Parsing ---
# Line 0: Header (NVM_V3)
# Line 1: Empty
# Line 2: Number of Cameras (Integer)
# Line 3: to 3+N: Camera Entries
# ... rest is point cloud data (we keep as is)

if len(lines) < 3:
    raise ValueError("NVM file is too short or corrupted.")

try:
    num_cams = int(lines[2].strip())
except ValueError:
    raise ValueError("Could not parse number of cameras from NVM file.")

print(f"Found {num_cams} cameras. Updating file paths...")

# Prepare new lines
new_lines = [lines[0], lines[1], lines[2]]  # Keep header and camera count

# Iterate only through the camera lines
for i in range(3, 3 + num_cams):
    parts = lines[i].split()
    original_filename = parts[0]
    
    # Strip any existing path info to be safe, then prepend the subdir
    clean_name = Path(original_filename).name 
    new_filename = f"{image_subdir}/{clean_name}"
    
    # Update the first part of the line
    parts[0] = new_filename
    
    # Reconstruct the line (NVM uses space separation)
    new_line = " ".join(parts) + "\n"
    new_lines.append(new_line)

# Append the rest of the file (3D points) unchanged
new_lines.extend(lines[3 + num_cams:])

# Overwrite the file
with open(scaled_model_path / "sfm_model_fixed.nvm", 'w') as f:
    f.writelines(new_lines)

logger.info(f"✅ Successfully updated {scaled_model_path / 'sfm_model_fixed.nvm'}")
logger.info(f"First image path is now: {new_lines[2].split()[0]}")

In [ ]:
# Texture Dense Scene

shutil.copyfile(paths["mvs_root"] / "scene_mesh_refined.ply", scaled_model_path / "scene_mesh_refined.ply")

command = [
    "texrecon",  
    "sfm_model_fixed.nvm", # in scene 
    "scene_mesh_refined.ply", # in mesh
    "textured_output", # out prefix
]

with OpenMVSEnv() as my_env:    
    run_cmd(command, cwd=str(scaled_model_path), env=my_env)

## 7. Pipeline Complete

**Outputs:**
1.  **Gaussian Splatting Source:** `output_root/sfm_superpoint_max+lightglue/geofenced` (contains sparse point cloud and cameras).
2.  **Dense Point Cloud:** `output_root/mvs_workspace/scene_dense.ply`
3.  **Textured Mesh:** `output_root/mvs_workspace/textured_model.glb`

In [ ]:
# Move textured output into a result directory
# All output with prefix "textured_output" is relevant 

textured_output_dir = Config.output_root / "textured_model"
textured_output_dir.mkdir(parents=True, exist_ok=True)

for f in scaled_model_path.glob("textured_output*"):
    shutil.move(str(f), str(textured_output_dir / f.name))
logger.info(f"Textured model and related files moved to: {textured_output_dir}")

logger.info("✅ Pipeline complete!")